# Synthetic tRBS cases, part 2: optimising and measuring the gap

**This is part 2 of 2.**
[Part 1](01_generating_synthetic_cases.ipynb) builds the cases and explains the
regimes. This part takes a generated case, optimises it, works out the
provably best allocation, and measures how far the optimiser was off.

## In one minute

The number this notebook produces is the **gap**:

    gap = f_star (the certified best score) - f_found (what the optimiser got)

A gap of zero means the optimiser found the best allocation possible. On a real
tRBS case this number cannot be computed at all, because `f_star` is unknown.
That is the whole reason the generated cases exist.

## 1. Setup

Same environment as part 1: the repository virtual environment with `vlinder`
installed editable, working directory `experiments/synthetic/`.

In [ ]:
import pandas as pd

import case_factory as cf
import oracle
from study_harness import StudyHarness, StudySpec

from vlinder.trbs import TheResponsibleBusinessSimulator
from vlinder.optimize import Optimize

# Note: vlinder.__init__ exposes only TheResponsibleBusinessSimulator and
# list_demo_cases, so Optimize is imported from vlinder.optimize directly.

## 2. Get a case

We rebuild the simple convex case from part 1. Same parameters and same seed
give byte-identical tables, so it does not matter whether part 1 has been run:
if the case is already on disk, this rewrites exactly what was there.

In [ ]:
params = cf.SyntheticCaseParams(k=3, n_key_outputs=3, seed=1)
root = cf.SyntheticCaseFactory(params).write()
print(f"Using case: {params.name}")

## 3. Run it through the real pipeline

Nothing here is specific to synthetic cases. This is the same sequence as
`vlinder_demo.ipynb`: point the simulator at the tables and run the pipeline.

In [ ]:
sim = TheResponsibleBusinessSimulator(params.name, file_path=root, file_extension="csv")
sim.build()
sim.evaluate()
sim.appreciate()
print("Pipeline complete: build, evaluate, appreciate")

## 4. Optimise with multi-start SLSQP

Now the part we are actually studying. SLSQP is a gradient-based solver: from a
starting allocation it walks uphill until it cannot improve. Because it stops at
the first peak it reaches, we restart it from 20 random points and keep the best
result.

The arguments:

- `scenario`: which scenario to optimise for.
- `budget`: the spending cap, passed explicitly rather than inferred.
- `dmo_name="SLSQP"`: the name the winning allocation is stored under. Naming it
  after the method keeps results apart when several methods run on one case.
- `n_starts=20`: how many random starting points to try.
- `seed=1`: makes those starting points reproducible.

The full study runs four methods (grid search, SLSQP, basin-hopping and a
genetic algorithm) through `Optimize.run(scenario, method=...)`. One is enough
to show the measurement.

In [ ]:
scenario = str(sim.input_dict["scenarios"][0])
print(f"Optimising scenario: {scenario}")

opt = Optimize(sim.input_dict, sim.output_dict)
slsqp = opt.optimize_slsqp(scenario, params.budget, dmo_name="SLSQP", n_starts=20, seed=1)

print(f"SLSQP appreciation: {slsqp.appreciation:.4f}")
print(f"Allocation (x):     {slsqp.allocation}")
print(f"Converged:          {slsqp.n_converged}/{slsqp.n_starts} starts")

## 5. Certify the answer with the oracle

`certify_case()` reads the regime from the manifest, picks the matching oracle,
computes the best possible allocation and writes it back into `manifest.json`.

For this case the appreciation is linear, so the score is proportional to how
much we put into each variable. The optimum then has to sit at a corner of the
feasible region. In plain terms: when everything is proportional, the best plan
is to put the entire budget into the single most rewarding variable. So it is
enough to check the three "all budget on one variable" plans plus the "spend
nothing" plan, and take the winner. The oracle does exactly that, and separately
computes the gradient by hand to confirm it points at the same corner.

In [ ]:
oracle_result = oracle.certify_case(params.name, root)

print(f"Oracle method: {oracle_result['method']}")
print(f"Verified:      {oracle_result['verified']}")
print(f"Oracle f* for {scenario}: {oracle_result['per_scenario'][scenario]['f_star']:.4f}")

### 5.1 What the oracle can and cannot guarantee

Worth being precise about, because the answer differs per case type. "Certain"
below means mathematically proven, not "we checked a lot of points".

| Case type | How the oracle finds the answer | How certain is it |
|---|---|---|
| convex, linear | Checks the k "all budget on one variable" plans plus "spend nothing" | **Certain**, at any size k |
| convex, curved | Tight local solver plus a KKT check | **Certain**: on a convex problem, a local optimum is the global one |
| non-convex, k <= 6 | Dense grid over the whole budget space, then polishing the best points | **Certain at grid resolution**: a peak narrower than the grid spacing could be missed |
| non-convex, k > 6 | Best of many local solves from different starting points | **Not certain**: this is a best effort, not a proof |

So for a non-convex case the oracle does **not** know the global optimum in
general. That is a real limit, not a detail. Two things follow from it, and both
are deliberate:

1. Above k = 6 the certificate is labelled `verified: false`. Numbers measured
   against it compare methods with each other, and are never read as "method X
   found the true optimum".
2. The thesis anchors its claim about absolute optimum recovery on the **convex**
   regime only, where the answer is provable at any problem size. For non-convex
   cases we report evidence about the shape of the landscape (how many peaks it
   has) rather than a recovery claim.

## 6. Did SLSQP find it?

The payoff. The **gap** is the distance between the certified best score and
what the optimiser actually achieved. A gap of zero means the optimiser found
the best allocation. On a real case this number cannot be computed at all.

In [ ]:
f_star = oracle_result["per_scenario"][scenario]["f_star"]
gap = f_star - slsqp.appreciation

print(f"f* (oracle): {f_star:.4f}")
print(f"f  (SLSQP):  {slsqp.appreciation:.4f}")
print(f"Gap:         {gap:.2e}")
print()
print("SLSQP recovered the certified optimum." if gap < 1e-6 else f"SLSQP missed it by {gap:.4f}.")

## 7. Cases with more than one peak

On the convex case any sensible optimiser wins, which is why it is the baseline
rather than the result. The research question starts where the landscape has
several peaks.

### 7.1 Smooth peaks

This is the `smooth_nonconvex` case from part 1. Because k <= 6, the oracle uses
the dense grid method from the table in section 5.1: it evaluates a fine grid
covering the whole budget space, then polishes the best points with a local
solver. It also estimates the number of **basins**. A basin is one valley in the
landscape, in the sense that a ball released anywhere inside it rolls to the
same bottom. More basins means more separate peaks, and more ways for an
optimiser to get stuck on the wrong one.

In [ ]:
params_smooth = cf.SyntheticCaseParams(
    k=3,
    n_key_outputs=3,
    regime="smooth_nonconvex",
    appreciation="sinusoidal",
    n_stb1=1,
    n_bilinear=1,
    seed=4,
)
root_smooth = cf.SyntheticCaseFactory(params_smooth).write()
oracle_smooth = oracle.certify_case(params_smooth.name, root_smooth)

cert = next(iter(oracle_smooth["per_scenario"].values()))["certificate"]
print(f"Case:                        {params_smooth.name}")
print(f"Oracle method:               {oracle_smooth['method']}")
print(f"Verified:                    {oracle_smooth['verified']}")
print(f"Grid resolution:             {cert.get('grid_resolution', 'n/a')}")
print(f"Estimated basins (by score): {cert.get('n_basins_f_estimate', 'n/a')}")
print(f"Estimated basins (by plan):  {cert.get('n_basins_x_estimate', 'n/a')}")

Notice the result: at this small size, curvature alone still leaves **one**
peak. That is worth stating plainly, because it is easy to assume that
"non-convex" automatically means "full of traps". It does not. Whether a bent
landscape actually acquires a second peak depends on how strongly it is bent and
on how many variables there are, which is exactly the thing this apparatus
measures instead of assuming.

### 7.2 Sharp edges, and a second peak

The `nonsmooth` case from part 1 is the one that does reliably split into two
peaks at this size. Narrowing the KPI boundaries creates flat plateaus with
sharp creases, and cutting off at the bottom of the scale is what splits a
single peak in two.

In [ ]:
params_sharp = cf.SyntheticCaseParams(
    k=3,
    n_key_outputs=3,
    regime="nonsmooth",
    bracketing_factor=0.6,
    seed=3,
)
root_sharp = cf.SyntheticCaseFactory(params_sharp).write()
oracle_sharp = oracle.certify_case(params_sharp.name, root_sharp)

cert_sharp = next(iter(oracle_sharp["per_scenario"].values()))["certificate"]
print(f"Case:                        {params_sharp.name}")
print(f"Estimated basins (by score): {cert_sharp.get('n_basins_f_estimate', 'n/a')}")
print(f"Estimated basins (by plan):  {cert_sharp.get('n_basins_x_estimate', 'n/a')}")

## 8. From one case to the scaling claim

A single case is an anecdote. The headline claim of the thesis is about
**scale**: as the number of internal variables k grows, does the optimiser still
find the true best allocation in reasonable time?

The full pre-registered study runs both convex variants over
k = 2, 3, 4, 6, 9, 12, 15 with 30 seeds each and four optimisation methods,
which is 5,040 individual runs. `StudyHarness` handles it: it generates each
case, certifies it, runs every method and records one row per result, resuming
where it left off if interrupted.

Below is a small version of the same thing, so it finishes while you watch.
The full design is fixed in `PREREGISTRATION.md`.

One thing to expect: the harness prints how many tasks it still has to run. The
first time you run this cell that number is 18. Run it again and it prints 0,
because every result is already on disk and it does not recompute them. That is
the same mechanism that let the real 5,040-run study survive being interrupted.

In [ ]:
spec = StudySpec(
    variants=("linear", "sinusoidal"),
    ks=(2, 3, 4),
    seeds=(0,),
    methods=("slsqp",),
    root=str(cf.DEFAULT_ROOT / "notebook_demo"),
)
harness = StudyHarness(spec)
results_path = harness.run(n_workers=1)

rows = pd.read_json(results_path, lines=True)
print()
print(rows[["case_name", "variant", "k", "method", "f_oracle", "gap", "recovered"]].to_string(index=False))

### 8.1 What that shows

Across every k tested, the gap on the convex-linear variant stays at the level
of numerical rounding error. That is the checkable version of the claim "the
method scales": not "it looked fine", but "it recovered the provably best
allocation at every size we tried".

The scope limit from section 5.1 applies here too. This absolute claim rests on
the convex regime, because that is where the answer is provable. For non-convex
cases we report how rugged the landscape is and verify at grid resolution for
small k, and we do not claim the global optimum was recovered.

## 9. Where to look next

- [Part 1: generating synthetic cases](01_generating_synthetic_cases.ipynb)
- [`case_factory.py`](case_factory.py): builds the cases.
- [`oracle.py`](oracle.py): computes the certified answers.
- [`study_harness.py`](study_harness.py): runs the full study grid.
- [`scip_benchmark.py`](scip_benchmark.py): the independent SCIP check on the
  certified answers.
- [`PREREGISTRATION.md`](PREREGISTRATION.md): the frozen study design.
- [`vlinder_demo.ipynb`](../../vlinder_demo.ipynb): the basics of tRBS itself.